In [3]:
                                                                                                                                                                                                                                                            # Transformers installation
! pip install transformers datasets evaluate accelerate
# To install from source instead of the last release, comment the command above and uncomment the following one.
# ! pip install git+https://github.com/huggingface/transformers.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


# Quickstart

Transformers is designed to be fast and easy to use so that everyone can start learning or building with transformer models.

The number of user-facing abstractions is limited to only three classes for instantiating a model, and two APIs for inference or training. This quickstart introduces you to Transformers' key features and shows you how to:

- load a pretrained model
- run inference with [Pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.Pipeline)
- fine-tune a model with [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer)

## Set up

To start, we recommend creating a Hugging Face [account](https://hf.co/join). An account lets you host and access version controlled models, datasets, and [Spaces](https://hf.co/spaces) on the Hugging Face [Hub](https://hf.co/docs/hub/index), a collaborative platform for discovery and building.

Create a [User Access Token](https://hf.co/docs/hub/security-tokens#user-access-tokens) and log in to your account.

<hfoptions id="authenticate">
<hfoption id="notebook">

Paste your User Access Token into [notebook_login](https://huggingface.co/docs/huggingface_hub/main/en/package_reference/authentication#huggingface_hub.notebook_login) when prompted to log in.

In [1]:
from huggingface_hub import notebook_login

notebook_login()

</hfoption>
<hfoption id="CLI">

Make sure the [huggingface_hub[cli]](https://huggingface.co/docs/huggingface_hub/guides/cli#getting-started) package is installed and run the command below. Paste your User Access Token when prompted to log in.

```bash
hf auth login
```

</hfoption>
</hfoptions>

Install Pytorch.

```bash
!pip install torch
```

Then install an up-to-date version of Transformers and some additional libraries from the Hugging Face ecosystem for accessing datasets and vision models, evaluating training, and optimizing training for large models.

```bash
!pip install -U transformers datasets evaluate accelerate timm
```

## Pretrained models

Each pretrained model inherits from three base classes.

| **Class** | **Description** |
|---|---|
| [PreTrainedConfig](https://huggingface.co/docs/transformers/main/en/main_classes/configuration#transformers.PreTrainedConfig) | A file that specifies a models attributes such as the number of attention heads or vocabulary size. |
| [PreTrainedModel](https://huggingface.co/docs/transformers/main/en/main_classes/model#transformers.PreTrainedModel) | A model (or architecture) defined by the model attributes from the configuration file. A pretrained model only returns the raw hidden states. For a specific task, use the appropriate model head to convert the raw hidden states into a meaningful result (for example, [LlamaModel](https://huggingface.co/docs/transformers/main/en/model_doc/llama2#transformers.LlamaModel) versus [LlamaForCausalLM](https://huggingface.co/docs/transformers/main/en/model_doc/llama2#transformers.LlamaForCausalLM)). |
| Preprocessor | A class for converting raw inputs (text, images, audio, multimodal) into numerical inputs to the model. For example, [PreTrainedTokenizer](https://huggingface.co/docs/transformers/main/en/main_classes/tokenizer#transformers.PreTrainedTokenizer) converts text into tensors and [ImageProcessingMixin](https://huggingface.co/docs/transformers/main/en/main_classes/image_processor#transformers.ImageProcessingMixin) converts pixels into tensors. |

We recommend using the [AutoClass](https://huggingface.co/docs/transformers/main/en/./model_doc/auto) API to load models and preprocessors because it automatically infers the appropriate architecture for each task and machine learning framework based on the name or path to the pretrained weights and configuration file.

Use [from_pretrained()](https://huggingface.co/docs/transformers/main/en/main_classes/model#transformers.PreTrainedModel.from_pretrained) to load the weights and configuration file from the Hub into the model and preprocessor class.

When you load a model, configure the following parameters to ensure the model is optimally loaded.

- `device_map="auto"` automatically allocates the model weights to your fastest device first.
- `dtype="auto"` directly initializes the model weights in the data type they're stored in, which can help avoid loading the weights twice (PyTorch loads weights in `torch.float32` by default).

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf", dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [10]:
model
model.device
#model.dtype
#model.cpu
#model.base_model_prefix

device(type='cuda', index=0)

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

Tokenize the text and return PyTorch tensors with the tokenizer. Move the model to an accelerator if it's available to accelerate inference.

In [4]:
model_inputs = tokenizer(["The secret to baking a good cake is "], return_tensors="pt").to(model.device)

NameError: name 'tokenizer' is not defined

The model is now ready for inference or training.

For inference, pass the tokenized inputs to [generate()](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation#transformers.GenerationMixin.generate) to generate text. Decode the token ids back into text with [batch_decode()](https://huggingface.co/docs/transformers/main/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode).

In [ ]:
generated_ids = model.generate(**model_inputs, max_length=30)
tokenizer.batch_decode(generated_ids)[0]
#'<s> The secret to baking a good cake is 100% in the preparation. There are so many recipes out there,'

> [!TIP]
> Skip ahead to the [Trainer](#trainer-api) section to learn how to fine-tune a model.

## Pipeline

The [Pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.Pipeline) class is the most convenient way to inference with a pretrained model. It supports many tasks such as text generation, image segmentation, automatic speech recognition, document question answering, and more.

> [!TIP]
> Refer to the [Pipeline](https://huggingface.co/docs/transformers/main/en/./main_classes/pipelines) API reference for a complete list of available tasks.

Create a [Pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.Pipeline) object and select a task. By default, [Pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.Pipeline) downloads and caches a default pretrained model for a given task. Pass the model name to the `model` parameter to choose a specific model.

<hfoptions id="pipeline-tasks">
<hfoption id="text generation">

Use `Accelerator` to automatically detect an available accelerator for inference.

In [6]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device

pipeline = pipeline("text-generation", model="meta-llama/Llama-2-7b-hf", device=device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


Prompt [Pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.Pipeline) with some initial text to generate more text.

In [7]:
pipeline("The secret to baking a good cake is ", max_length=50)
#[{'generated_text': 'The secret to baking a good cake is 100% in the batter. The secret to a great cake is the icing.\nThis is why we’ve created the best buttercream frosting reci'}]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


[{'generated_text': "The secret to baking a good cake is 1/3rd of a teaspoon of baking powder. It's the same with being a good leader.\nA good leader is one who knows how to motiv"}]

In [8]:
from accelerate import Accelerator

device = Accelerator().device
print(device)

cuda


</hfoption>
<hfoption id="image segmentation">

Use `Accelerator` to automatically detect an available accelerator for inference.

In [2]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device
print(device)

pipeline = pipeline("image-segmentation", model="facebook/detr-resnet-50-panoptic", device=device)

cuda


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/172M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/172M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2441: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pas

preprocessor_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda


Pass an image - a URL or local path to the image - to [Pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.Pipeline).

<div class="flex justify-center">
   <img src="https://huggingface.co/datasets/Narsil/image_dummy/raw/main/parrots.png"/>
</div>

In [9]:
segments = pipeline("https://huggingface.co/datasets/Narsil/image_dummy/raw/main/parrots.png")
print(segments[0])
print(segments[1])
#'bird'
##segments[1]["label"]
#'bird'

{'score': 0.999439, 'label': 'bird', 'mask': <PIL.Image.Image image mode=L size=768x512 at 0x7865C17764B0>}
{'score': 0.998787, 'label': 'bird', 'mask': <PIL.Image.Image image mode=L size=768x512 at 0x786663517740>}


</hfoption>
<hfoption id="automatic speech recognition">

Use `Accelerator` to automatically detect an available accelerator for inference.

In [10]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device
print(device)
pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3", device=device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Device set to use cuda


In [18]:
pipeline("/sadachar_21-11-25.mp4")

ValueError: Soundfile is either not in the correct format or is malformed. Ensure that the soundfile has a valid audio file extension (e.g. wav, flac or mp3) and is not corrupted. If reading from a remote URL, ensure that the URL is the full address to **download** the audio file.

In [40]:
input_mp4_file = '/sadachar_21-11-25.mp4'
output_wav_file = 'output.wav'

# -ar 16000 sets the audio sampling rate to 16kHz
# -t 20 limits the output duration to 20 seconds
!ffmpeg -i {input_mp4_file} -vn -acodec pcm_s16le -ar 16000 -ac 1 -t 200 {output_wav_file}

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Pass an audio file to [Pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.Pipeline).

In [33]:

pipeline("output.wav",return_timestamps=True)
#{'text': ' He hoped there would be stew for dinner, turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick, peppered flour-fatten sauce.'}

Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


{'text': ' शीराम् जयराम् जयजयराम् शीराम् जयराम् जयजयराम् शीराम् जयराम् जयजयराम् ओम् श्री हनुमते नमः। ओम् श्री सरस्वत्यै नमः। प्रिय आत्मान। सदाचार की वेला प्रारंब हो गई। और समय बताने के आवश्यक्ता नहीं लेकिन इधर बता रहा हूँ तो बता देता हूँ पाँच पच्करुन्तालिस्ट मिनट अब है। आज शुक्रवार है। कल शनिवार परसुन रविवार। हम लोगों का एक उत्साह है कि छोटी सी बैठक कर रहे हैं हम लोग लखनव में। वो सम्मेल नहीं है कि इनको नहीं बुलाया, उनको नहीं बुलाया, उनको नहीं बुलाया। जिनको बुलाया है वो विचार करके वहाँ पहुँचें। और अभी उन्होंने स्थान नहीं बताया है हमारे पुनित जी ने। बता देंगे। और पहुँचना है, सुनना है, गुनना है, और जो मन में आये उसको व्यक्त भी करना है। बैठकों में जब विशयानतर होने लगते हैं, तो फिर समय एक प्रकार से बरबाद होता है। सबको लगता है कि मैं भी कुछ कहूँ, मैं भी कुछ कहूँ, लेकिन जो संघ के कारिकर्ता हैं, अब वो बुजुर्ग हो गए हैं, नई पीढ़ी उतना सीख नहीं पाई है, उन नियमों को, तो उनसे सीखना चाहिए, वो बैठके होती थी, जो बैठक लेने वाला है, वो बोल रहा है, बोल रहा है, बोल रहा है, और कुछ बैठके तो ऐसी होती थी किव

In [35]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device
print(device)

# Using a faster, distilled version of Whisper
pipeline_fast = pipeline("automatic-speech-recognition", model="distil-whisper/distil-large-v3", device=device)

cuda


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.51G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Device set to use cuda


In [36]:
result = pipeline_fast("output.wav", return_timestamps=True)

print("Full Transcript:", result['text'])
print("\nStreaming Chunks (simulated):")
for chunk in result['chunks']:
    print(f"[{chunk['timestamp'][0]:.2f}s - {chunk['timestamp'][1]:.2f}s] {chunk['text']}")

Full Transcript:  Shiraam, Jaya Ram, Jaya Jaya Ram. Shiraam, Jaya Ram, Jaya Jaya Ram. Shiraam, Jaya Ram. Shiraam, Jaya Ram, Jaya Jaya Ram. O Ome, Shri Hanumatei Namaha. Om Shri Saraswati, Namaha. Preehātman. SADHARC-Ked Vela Phran Bhaarabh. And the time I'm not, but I'm here I'm, I'll tell them, I'll let's say I'm, 5.5.49 minute, now is. Today Shukrwara is. Kall, Shan-I-War, Purson, Rhabi-War. We are A lot of A little bit of a little bit of a little bit of in Lakhnau in That's the Mellin not, that they're not called, they've not called and they've called. Jinn to have they've vichar and we're there come to and they're not they have our Punijee and he will and to get them to be. And......ponchna, listen, and gunna, and what in the mind in it is, to make, when when when vishayantar are going to then, then time is a kind of It is a bit of a bit of a bit of something, I also, I also can't, but the SANGHs, but the SANGs who are, now, they've been the new people, they've got to learn that th

Now you can use `pipeline_fast` with your `output.wav` file:

In [41]:
#pipeline("output.wav", return_timestamps=True)
pipeline("output.wav",return_timestamps=True)

KeyError: "Unknown task output.wav, available tasks are ['audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'image-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'summarization', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'text2text-generation', 'token-classification', 'translation', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

</hfoption>
</hfoptions>

## Trainer

[Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) is a complete training and evaluation loop for PyTorch models. It abstracts away a lot of the boilerplate usually involved in manually writing a training loop, so you can start training faster and focus on training design choices. You only need a model, dataset, a preprocessor, and a data collator to build batches of data from the dataset.

Use the [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments) class to customize the training process. It provides many options for training, evaluation, and more. Experiment with training hyperparameters and features like batch size, learning rate, mixed precision, torch.compile, and more to meet your training needs. You could also use the default training parameters to quickly produce a baseline.

Load a model, tokenizer, and dataset for training.

In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset

model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
dataset = load_dataset("rotten_tomatoes")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

To use your previously trained `navinrathore/distilbert-rotten-tomatoes` model with the `pipeline`, you should specify the task as `"text-classification"`.

In [67]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device
print(device)

# Initialize the pipeline for text classification with your fine-tuned model
classifier_pipeline = pipeline("fill-mask", model="navinrathore/distilbert-rotten-tomatoes", device=device)

Some weights of DistilBertForMaskedLM were not initialized from the model checkpoint at navinrathore/distilbert-rotten-tomatoes and are newly initialized: ['vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.bias', 'vocab_transform.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cuda


Device set to use cuda


In [68]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device
print(device)

# Initialize the pipeline for text classification with your fine-tuned model
classifier_pipeline_base = pipeline("fill-mask", model=model, tokenizer=tokenizer, device=device)

Device set to use cuda


cuda


Now you can use this `classifier_pipeline` to classify new text. The output will typically include a label (e.g., 'LABEL_0' for negative, 'LABEL_1' for positive based on the Rotten Tomatoes dataset) and a score indicating the confidence.

In [70]:
example_text = "This movie was extremly [MASK], a must-watch!"
result = classifier_pipeline(example_text)
print(result)

example_text_2 = "I am down today with [MASK]. Not feeling like doing anything"
result_2 = classifier_pipeline(example_text_2)
print(result_2)

[{'score': 0.0018348390003666282, 'token': 2345, 'token_str': 'final', 'sequence': 'this movie was extremly final, a must - watch!'}, {'score': 0.0014106534654274583, 'token': 24240, 'token_str': 'queue', 'sequence': 'this movie was extremly queue, a must - watch!'}, {'score': 0.0013488003751263022, 'token': 9013, 'token_str': '##ien', 'sequence': 'this movie was extremlyien, a must - watch!'}, {'score': 0.0012736968928948045, 'token': 2308, 'token_str': 'women', 'sequence': 'this movie was extremly women, a must - watch!'}, {'score': 0.0010062531800940633, 'token': 17214, 'token_str': 'tam', 'sequence': 'this movie was extremly tam, a must - watch!'}]
[{'score': 0.002709729829803109, 'token': 24562, 'token_str': '##dora', 'sequence': 'i am down today withdora. not feeling like doing anything'}, {'score': 0.0026836178731173277, 'token': 25290, 'token_str': '##zai', 'sequence': 'i am down today withzai. not feeling like doing anything'}, {'score': 0.00195228960365057, 'token': 27360, 't

In [42]:
#my own tests
example_text_2 = "I am feeling great [MASK]. Succesfully runnning the own trained model"
result_3 = classifier_pipeline(example_text_2)
print(result_3)
result_4 = classifier_pipeline_base(example_text_2)
print(result_4)

[{'score': 0.001717211096547544, 'token': 24729, 'token_str': '##loh', 'sequence': 'i am feeling greatloh. succesfully runnning the own trained model'}, {'score': 0.0014347918331623077, 'token': 24116, 'token_str': 'dissent', 'sequence': 'i am feeling great dissent. succesfully runnning the own trained model'}, {'score': 0.0012948462972417474, 'token': 7100, 'token_str': '##hill', 'sequence': 'i am feeling greathill. succesfully runnning the own trained model'}, {'score': 0.0012674612225964665, 'token': 22269, 'token_str': '##hp', 'sequence': 'i am feeling greathp. succesfully runnning the own trained model'}, {'score': 0.0012089102528989315, 'token': 24869, 'token_str': 'bn', 'sequence': 'i am feeling great bn. succesfully runnning the own trained model'}]


IndexError: too many indices for tensor of dimension 2

In [64]:
# example_text = "This movie was absolutely [MASK], a must-watch!"
# result = classifier_pipeline_base(example_text)
# print(result)

example_text_2 = "The quick brown fox [MASK] over the lazy dog."
#example_text_2 = "The plot was uninspired and the acting fell flat."
result_2 = classifier_pipeline_base(example_text_2)
print(result_2)

IndexError: too many indices for tensor of dimension 2

Create a function to tokenize the text and convert it into PyTorch tensors. Apply this function to the whole dataset with the [map](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.map) method.

In [2]:
def tokenize_dataset(dataset):
    return tokenizer(dataset["text"])
dataset = dataset.map(tokenize_dataset, batched=True)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Load a data collator to create batches of data and pass the tokenizer to it.

In [3]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Next, set up [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments) with the training features and hyperparameters.

In [5]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="distilbert-rotten-tomatoes",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    push_to_hub=True,
)

Finally, pass all these separate components to [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) and call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train) to start.

In [6]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

/tmp/ipython-input-2544145158.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: navin-rathore (navin-rathore-selflearner) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.454600
1000,0.394200
1500,0.264100
2000,0.289000
2500,0.187100
3000,0.138900
3500,0.104800
4000,0.087000


TrainOutput(global_step=4268, training_loss=0.229461462562511, metrics={'train_runtime': 631.4381, 'train_samples_per_second': 54.035, 'train_steps_per_second': 6.759, 'total_flos': 391995352809576.0, 'train_loss': 0.229461462562511, 'epoch': 4.0})

Share your model and tokenizer to the Hub with [push_to_hub()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.push_to_hub).

In [7]:
trainer.push_to_hub()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...omatoes/training_args.bin: 100%|##########| 5.84kB / 5.84kB            

  ...4239.010c23018047.19264.0: 100%|##########| 6.96kB / 6.96kB            

  ...omatoes/model.safetensors:  13%|#2        | 33.5MB /  268MB            

CommitInfo(commit_url='https://huggingface.co/navinrathore/distilbert-rotten-tomatoes/commit/f17aaa83c90062e34b692d3cc3babc43734bda0a', commit_message='End of training', commit_description='', oid='f17aaa83c90062e34b692d3cc3babc43734bda0a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/navinrathore/distilbert-rotten-tomatoes', endpoint='https://huggingface.co', repo_type='model', repo_id='navinrathore/distilbert-rotten-tomatoes'), pr_revision=None, pr_num=None)

First, let's create a new text generation pipeline using a different model, for example, `distilgpt2`. This model is much smaller than Llama-2 and will download and load faster.

In [65]:
from transformers import pipeline
from accelerate import Accelerator

device = Accelerator().device
print(device)

# Initialize a new pipeline with 'distilgpt2' for text generation
text_generator = pipeline("fill-mask", model="navinrathore/distilbert-rotten-tomatoes", device=device)

Some weights of DistilBertForMaskedLM were not initialized from the model checkpoint at navinrathore/distilbert-rotten-tomatoes and are newly initialized: ['vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.bias', 'vocab_transform.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cuda


Device set to use cuda


Now, let's use this new `text_generator` pipeline to generate some text with an example input.

In [66]:
example_text = "The quick brown fox [MASK] over the lazy dog."
result = text_generator(example_text)
result

[{'score': 0.0029887801501899958,
  'token': 28551,
  'token_str': '##rued',
  'sequence': 'the quick brown foxrued over the lazy dog.'},
 {'score': 0.0023526528384536505,
  'token': 9527,
  'token_str': '##dom',
  'sequence': 'the quick brown foxdom over the lazy dog.'},
 {'score': 0.0021860511042177677,
  'token': 21476,
  'token_str': '##rangle',
  'sequence': 'the quick brown foxrangle over the lazy dog.'},
 {'score': 0.0019299158593639731,
  'token': 21958,
  'token_str': '##jong',
  'sequence': 'the quick brown foxjong over the lazy dog.'},
 {'score': 0.0018599900649860501,
  'token': 25013,
  'token_str': '##promising',
  'sequence': 'the quick brown foxpromising over the lazy dog.'}]

Congratulations, you just trained your first model with Transformers!

## Next steps

Now that you have a better understanding of Transformers and what it offers, it's time to keep exploring and learning what interests you the most.

- **Base classes**: Learn more about the configuration, model and processor classes. This will help you understand how to create and customize models, preprocess different types of inputs (audio, images, multimodal), and how to share your model.
- **Inference**: Explore the [Pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.Pipeline) further, inference and chatting with LLMs, agents, and how to optimize inference with your machine learning framework and hardware.
- **Training**: Study the [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) in more detail, as well as distributed training and optimizing training on specific hardware.
- **Quantization**: Reduce memory and storage requirements with quantization and speed up inference by representing weights with fewer bits.
- **Resources**: Looking for end-to-end recipes for how to train and inference with a model for a specific task? Check out the task recipes!